In [2]:
import os
import json
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

# -------- Dataset：260次元特徴 + 相対速度 --------
class RelativeSpeedDataset260D(Dataset):
    def __init__(self, annot_root, distance_json_path, max_items=None):
        self.items = []

        print("📥 距離ファイル読み込み中...")
        with open(distance_json_path, encoding='utf-8') as f:
            self.distances = json.load(f)

        for fname in sorted(os.listdir(annot_root)):
            if not fname.endswith(".json"):
                continue

            sid = fname.replace(".json", "")
            print(f"📂 処理中: {sid}")

            if sid not in self.distances:
                print(f"❌ スキップ: 距離情報なし")
                continue

            with open(os.path.join(annot_root, fname), encoding='utf-8') as f:
                ann = json.load(f)

            seq = ann.get("sequence", [])
            if len(seq) < 20:
                print(f"⚠️ スキップ: フレーム数 {len(seq)} 未満")
                continue

            own = np.array([f['OwnSpeed'] for f in seq], dtype=np.float32)
            tgt = np.array([f['TgtSpeed_ref'] for f in seq], dtype=np.float32)
            keys = [f"frame_{i+1:05d}" for i in range(len(seq))]
            dist = np.array([self.distances[sid].get(k, np.nan) for k in keys], dtype=np.float32)

            if len(dist) < 20:
                print(f"⚠️ スキップ: 距離データが20未満（{len(dist)}）")
                continue

            def smooth(x, w):
                if len(x) < w:
                    return np.zeros_like(x)
                return np.convolve(x, np.ones(w)/w, mode='same')

            for i in range(len(seq) - 19):
                if max_items and len(self.items) >= max_items:
                    return

                d = dist[i:i+20]
                o = own[i:i+20]
                t = tgt[i:i+20]

                if np.any(np.isnan(d)) or np.any(np.isnan(o)) or np.any(np.isnan(t)):
                    continue

                rel_speed = t - o
                own_acc = np.gradient(o)
                d1 = np.gradient(d)
                d2 = np.gradient(d1)

                f3 = smooth(d, 3)
                f5 = smooth(d, 5)
                f7 = smooth(d, 7)
                f11 = smooth(d, 11)
                f11_d1 = np.gradient(f11) if len(f11) >= 3 else np.zeros_like(f11)

                try:
                    feat = np.stack([
                        d[:20], o[:20], own_acc[:20], d1[:20], d2[:20],
                        f3[:20], f5[:20], f7[:20], f11[:20], f11_d1[:20],
                        f3[:20] * d1[:20], f11[:20] - f5[:20], np.abs(d1[:20])
                    ], axis=1)
                except Exception as e:
                    print(f"❌ 特徴量結合エラー @ {sid} frame {i}: {e}")
                    continue

                if feat.shape != (20, 13):
                    continue

                target = np.mean(rel_speed)
                self.items.append((feat.astype(np.float32), target, sid))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        feat, tgt, sid = self.items[idx]
        return feat, tgt, sid


# -------- 改良版 LSTM モデル --------
class LSTM260D_v2(nn.Module):
    def __init__(self, input_size=13, hidden_size=256, num_layers=3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.3
        )
        self.fc = nn.Sequential(
            nn.BatchNorm1d(hidden_size),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 1)
        )

    def forward(self, x):  # x: (B, T, F)
        out, _ = self.lstm(x)
        pooled = out.mean(dim=1)  # 時系列方向に平均 (B, H)
        return self.fc(pooled).squeeze(1)


# -------- 単一分割での学習ループ --------
def train_single_split(dataset, save_path="model_lstm260d_v2.pth"):
    scenes = sorted(set([item[-1] for item in dataset.items]))
    train_scenes, val_scenes = train_test_split(scenes, test_size=0.2, random_state=42)
    train_ds = [item for item in dataset.items if item[-1] in train_scenes]
    val_ds = [item for item in dataset.items if item[-1] in val_scenes]

    def collate_fn(batch):
        feats, tgts, sids = zip(*batch)
        feats = [torch.tensor(f, dtype=torch.float32) for f in feats]
        return torch.stack(feats), torch.tensor(tgts, dtype=torch.float32), list(sids)

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, collate_fn=collate_fn)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = LSTM260D_v2().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.3,
        patience=3
    )

    criterion = nn.SmoothL1Loss()

    best_val_loss = float('inf')
    patience = 30
    counter = 0

    for epoch in range(50):
        model.train()
        total_train_loss = 0
        for feats, tgts, _ in tqdm(train_loader, desc=f"[Train Epoch {epoch+1}]"):
            feats, tgts = feats.to(device), tgts.to(device)
            pred = model(feats)
            loss = criterion(pred, tgts)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item() * feats.size(0)

        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for feats, tgts, _ in val_loader:
                feats, tgts = feats.to(device), tgts.to(device)
                pred = model(feats)
                loss = criterion(pred, tgts)
                total_val_loss += loss.item() * feats.size(0)

        train_loss = total_train_loss / len(train_ds)
        val_loss = total_val_loss / len(val_ds)
        scheduler.step(val_loss)

        print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), save_path)
            print(f"✅ モデル保存: {save_path}（val_loss={val_loss:.4f}）")
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                print("🛑 Early Stopping")
                break


# -------- 実行部 --------
if __name__ == "__main__":
    dataset = RelativeSpeedDataset260D(
        annot_root="./train_annotations",
        distance_json_path="../train2/filtered_spline_smoothed.json",
        max_items=40000
    )
    print(f"✅ dataset loaded: {len(dataset)} samples")
    train_single_split(dataset, save_path="model_lstm260d_v2.pth")


📥 距離ファイル読み込み中...
📂 処理中: 000
📂 処理中: 001
❌ スキップ: 距離情報なし
📂 処理中: 002
📂 処理中: 003
📂 処理中: 004
📂 処理中: 005
📂 処理中: 006
📂 処理中: 007
📂 処理中: 008
📂 処理中: 009
📂 処理中: 010
📂 処理中: 011
📂 処理中: 012
❌ スキップ: 距離情報なし
📂 処理中: 013
📂 処理中: 014
📂 処理中: 015
📂 処理中: 016
📂 処理中: 017
📂 処理中: 018
📂 処理中: 019
📂 処理中: 020
📂 処理中: 021
📂 処理中: 022
📂 処理中: 023
📂 処理中: 024
📂 処理中: 025
📂 処理中: 026
📂 処理中: 027
📂 処理中: 028
📂 処理中: 029
📂 処理中: 030
📂 処理中: 031
📂 処理中: 032
📂 処理中: 033
📂 処理中: 034
📂 処理中: 035
📂 処理中: 036
📂 処理中: 037
📂 処理中: 038
📂 処理中: 039
📂 処理中: 040
📂 処理中: 041
📂 処理中: 042
📂 処理中: 043
📂 処理中: 044
📂 処理中: 045
📂 処理中: 046
📂 処理中: 047
📂 処理中: 048
📂 処理中: 049
📂 処理中: 050
📂 処理中: 051
📂 処理中: 052
📂 処理中: 053
📂 処理中: 054
📂 処理中: 055
📂 処理中: 056
📂 処理中: 057
📂 処理中: 058
📂 処理中: 059
📂 処理中: 060
📂 処理中: 061
📂 処理中: 062
📂 処理中: 063
📂 処理中: 064
📂 処理中: 065
📂 処理中: 066
📂 処理中: 067
📂 処理中: 068
📂 処理中: 069
📂 処理中: 070
📂 処理中: 071
📂 処理中: 072
📂 処理中: 073
📂 処理中: 074
📂 処理中: 075
📂 処理中: 076
📂 処理中: 077
📂 処理中: 078
📂 処理中: 079
📂 処理中: 080
📂 処理中: 081
📂 処理中: 082
📂 処理中: 083
📂 処理中: 084
📂 処理中: 085
📂 処理中: 

[Train Epoch 1]: 100%|██████████| 506/506 [00:03<00:00, 141.27it/s]


Epoch 1 | Train Loss: 0.6638 | Val Loss: 0.4346
✅ モデル保存: model_lstm260d_v2.pth（val_loss=0.4346）


[Train Epoch 2]: 100%|██████████| 506/506 [00:03<00:00, 144.49it/s]


Epoch 2 | Train Loss: 0.5644 | Val Loss: 0.1925
✅ モデル保存: model_lstm260d_v2.pth（val_loss=0.1925）


...

Epoch 48 | Train Loss: 0.1965 | Val Loss: 0.0673


[Train Epoch 49]: 100%|██████████| 506/506 [00:03<00:00, 149.02it/s]


Epoch 49 | Train Loss: 0.1969 | Val Loss: 0.0667


[Train Epoch 50]: 100%|██████████| 506/506 [00:03<00:00, 148.61it/s]


Epoch 50 | Train Loss: 0.1929 | Val Loss: 0.0670
